# 05 — EcoSpold workflow without a Brightway project

**Audience:** Users who have ecoinvent EcoSpold files but do not want to create or activate a Brightway project.

**Prerequisites:** A licensed ecoinvent `datasets/` directory, a valid IAM key, and `premise` installed.

**Learning goals:** read EcoSpold directly, apply Premise transformations, and export to a non-Brightway format.


## Outline

1. Read configuration from the environment.
2. Build from EcoSpold without project setup.
3. Choose one non-Brightway export.


In [ ]:
import os
from pathlib import Path

from premise import NewDatabase

ecospold_value = os.environ.get("ECOSPOLD_DIR")
if not ecospold_value:
    raise RuntimeError("Set ECOSPOLD_DIR to the ecoinvent datasets directory.")

ECOSPOLD_DIR = Path(ecospold_value).expanduser()
SOURCE_VERSION = "3.12"
PREMISE_KEY = os.environ.get("PREMISE_KEY")
EXPORT_ROOT = Path("export/ecospold-without-brightway")
EXPORT_KIND = "matrices"  # matrices, simapro, or olca

if not ECOSPOLD_DIR.is_dir():
    raise FileNotFoundError(ECOSPOLD_DIR)
if not PREMISE_KEY:
    raise RuntimeError("Set PREMISE_KEY before running this tutorial.")


## 1. Build from EcoSpold

There is deliberately no `bw2data.projects.set_current(...)` call. A Brightway biosphere database is only needed if you later choose a Brightway export.


In [ ]:
SCENARIOS = [
    {"model": "image", "pathway": "SSP2-M", "year": 2030},
]

ndb = NewDatabase(
    scenarios=SCENARIOS,
    source_type="ecospold",
    source_file_path=str(ECOSPOLD_DIR),
    source_version=SOURCE_VERSION,
    key=PREMISE_KEY,
)

TRANSFORMATIONS = None  # Example: ["electricity"]
if TRANSFORMATIONS is None:
    ndb.update()
else:
    ndb.update(TRANSFORMATIONS)


## 2. Export without Brightway

Choose exactly one branch. Premise creates missing output directories.


In [ ]:
if EXPORT_KIND == "matrices":
    output = EXPORT_ROOT / "matrices"
    ndb.write_db_to_matrices(filepath=str(output))
elif EXPORT_KIND == "simapro":
    output = EXPORT_ROOT / "simapro"
    ndb.write_db_to_simapro(filepath=str(output))
elif EXPORT_KIND == "olca":
    output = EXPORT_ROOT / "olca"
    ndb.write_db_to_olca(filepath=str(output))
else:
    raise ValueError(f"Unsupported EXPORT_KIND: {EXPORT_KIND}")

output


## Pitfalls and extension

- `ECOSPOLD_DIR` must point to the directory containing the dataset XML files, not its parent.
- `SOURCE_VERSION` must match those files.
- Do not configure a Brightway biosphere merely for matrix, SimaPro, or OpenLCA export.

## Exercise

Configure a targeted electricity-only update and a SimaPro export without changing the build cell.


In [ ]:
exercise_transformations = ["electricity"]
exercise_export_kind = "simapro"
